<a href="https://colab.research.google.com/github/KeerthanaSistla/BigDataAnalytics/blob/main/BDA_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Advanced Hive Simulation").getOrCreate()

#HIVE CASE STUDY

In [21]:
data = [
    (1, "Keerthana", "Math", 85, "Sem1", "Internal"),
    (1, "Keerthana", "Physics", 78, "Sem1", "External"),
    (2, "Rahul", "Math", 45, "Sem1", "Internal"),
    (2, "Rahul", "Physics", 55, "Sem1", "External"),
    (3, "Anjali", "Math", 92, "Sem2", "Internal"),
    (3, "Anjali", "Physics", 88, "Sem2", "External"),
    (4, "Arjun", "Math", 30, "Sem2", "Internal"),
    (4, "Arjun", "Physics", 40, "Sem2", "External")
]

columns = ["student_id", "name", "subject", "marks", "semester", "exam_type"]

df = spark.createDataFrame(data, columns)
df.show()

+----------+---------+-------+-----+--------+---------+
|student_id|     name|subject|marks|semester|exam_type|
+----------+---------+-------+-----+--------+---------+
|         1|Keerthana|   Math|   85|    Sem1| Internal|
|         1|Keerthana|Physics|   78|    Sem1| External|
|         2|    Rahul|   Math|   45|    Sem1| Internal|
|         2|    Rahul|Physics|   55|    Sem1| External|
|         3|   Anjali|   Math|   92|    Sem2| Internal|
|         3|   Anjali|Physics|   88|    Sem2| External|
|         4|    Arjun|   Math|   30|    Sem2| Internal|
|         4|    Arjun|Physics|   40|    Sem2| External|
+----------+---------+-------+-----+--------+---------+



In [22]:
df.write.partitionBy("semester", "exam_type") \
    .mode("overwrite") \
    .csv("advanced_partitioned_data")

In [23]:
df = df.withColumn("performance",
    when(col("marks") >= 85, "Excellent")
    .when(col("marks") >= 60, "Good")
    .when(col("marks") >= 40, "Average")
    .otherwise("At Risk")
)

df.show()

+----------+---------+-------+-----+--------+---------+-----------+
|student_id|     name|subject|marks|semester|exam_type|performance|
+----------+---------+-------+-----+--------+---------+-----------+
|         1|Keerthana|   Math|   85|    Sem1| Internal|  Excellent|
|         1|Keerthana|Physics|   78|    Sem1| External|       Good|
|         2|    Rahul|   Math|   45|    Sem1| Internal|    Average|
|         2|    Rahul|Physics|   55|    Sem1| External|    Average|
|         3|   Anjali|   Math|   92|    Sem2| Internal|  Excellent|
|         3|   Anjali|Physics|   88|    Sem2| External|  Excellent|
|         4|    Arjun|   Math|   30|    Sem2| Internal|    At Risk|
|         4|    Arjun|Physics|   40|    Sem2| External|    Average|
+----------+---------+-------+-----+--------+---------+-----------+



In [24]:
df.groupBy("performance").count().show()

+-----------+-----+
|performance|count|
+-----------+-----+
|  Excellent|    3|
|    Average|    3|
|       Good|    1|
|    At Risk|    1|
+-----------+-----+



In [25]:
from pyspark.sql.window import Window

window = Window.partitionBy("semester").orderBy(desc("marks"))

topper = df.withColumn("rank", row_number().over(window))
topper.filter("rank = 1").show()

+----------+---------+-------+-----+--------+---------+-----------+----+
|student_id|     name|subject|marks|semester|exam_type|performance|rank|
+----------+---------+-------+-----+--------+---------+-----------+----+
|         1|Keerthana|   Math|   85|    Sem1| Internal|  Excellent|   1|
|         3|   Anjali|   Math|   92|    Sem2| Internal|  Excellent|   1|
+----------+---------+-------+-----+--------+---------+-----------+----+



In [26]:
df.groupBy("semester", "subject") \
  .avg("marks") \
  .orderBy("semester") \
  .show()

+--------+-------+----------+
|semester|subject|avg(marks)|
+--------+-------+----------+
|    Sem1|Physics|      66.5|
|    Sem1|   Math|      65.0|
|    Sem2|   Math|      61.0|
|    Sem2|Physics|      64.0|
+--------+-------+----------+



In [27]:
df.filter(df.performance == "At Risk").show()

+----------+-----+-------+-----+--------+---------+-----------+
|student_id| name|subject|marks|semester|exam_type|performance|
+----------+-----+-------+-----+--------+---------+-----------+
|         4|Arjun|   Math|   30|    Sem2| Internal|    At Risk|
+----------+-----+-------+-----+--------+---------+-----------+



#SPARK SQL CASE STUDY

In [28]:
data = [
    (1, "Laptop", "Electronics", 50000, 2, "Premium"),
    (2, "Phone", "Electronics", 20000, 5, "Regular"),
    (3, "Shirt", "Clothing", 1500, 10, "Regular"),
    (4, "Shoes", "Footwear", 3000, 4, "Premium"),
    (5, "Watch", "Accessories", 7000, 3, "Premium"),
    (6, "Bag", "Accessories", 2500, 6, "Regular")
]

columns = ["order_id", "product", "category", "price", "quantity", "customer_type"]

df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("sales")
df.show()

+--------+-------+-----------+-----+--------+-------------+
|order_id|product|   category|price|quantity|customer_type|
+--------+-------+-----------+-----+--------+-------------+
|       1| Laptop|Electronics|50000|       2|      Premium|
|       2|  Phone|Electronics|20000|       5|      Regular|
|       3|  Shirt|   Clothing| 1500|      10|      Regular|
|       4|  Shoes|   Footwear| 3000|       4|      Premium|
|       5|  Watch|Accessories| 7000|       3|      Premium|
|       6|    Bag|Accessories| 2500|       6|      Regular|
+--------+-------+-----------+-----+--------+-------------+



In [29]:
spark.sql("""
SELECT SUM(price * quantity) AS total_revenue FROM sales
""").show()

+-------------+
|total_revenue|
+-------------+
|       263000|
+-------------+



In [30]:
spark.sql("""
SELECT category,
       SUM(price * quantity) as revenue,
       ROUND(100 * SUM(price * quantity) /
       SUM(SUM(price * quantity)) OVER (), 2) as percentage
FROM sales
GROUP BY category
""").show()

+-----------+-------+----------+
|   category|revenue|percentage|
+-----------+-------+----------+
|Electronics| 200000|     76.05|
|   Clothing|  15000|       5.7|
|   Footwear|  12000|      4.56|
|Accessories|  36000|     13.69|
+-----------+-------+----------+



In [31]:
spark.sql("""
SELECT customer_type,
       AVG(price * quantity) as avg_spending
FROM sales
GROUP BY customer_type
""").show()

+-------------+------------------+
|customer_type|      avg_spending|
+-------------+------------------+
|      Premium|44333.333333333336|
|      Regular|43333.333333333336|
+-------------+------------------+



In [32]:
spark.sql("""
SELECT product, revenue FROM (
    SELECT product,
           SUM(price * quantity) as revenue,
           RANK() OVER (ORDER BY SUM(price * quantity) DESC) as rank
    FROM sales
    GROUP BY product
) t WHERE rank = 1
""").show()

+-------+-------+
|product|revenue|
+-------+-------+
|  Phone| 100000|
| Laptop| 100000|
+-------+-------+



In [33]:
spark.sql("""
SELECT * FROM sales
WHERE price * quantity > 20000
""").show()

+--------+-------+-----------+-----+--------+-------------+
|order_id|product|   category|price|quantity|customer_type|
+--------+-------+-----------+-----+--------+-------------+
|       1| Laptop|Electronics|50000|       2|      Premium|
|       2|  Phone|Electronics|20000|       5|      Regular|
|       5|  Watch|Accessories| 7000|       3|      Premium|
+--------+-------+-----------+-----+--------+-------------+

